In [ ]:
import scanpy as sc
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy
import pandas as pd
import decoupler as dc
from scipy.sparse import csr_matrix
import os 
sc.set_figure_params(figsize=(4, 4))

date = "DATE"

In [ ]:
base_path = '/home/EOCRC_atlas/'

In [ ]:
# load in Tier2 annotated adata 
adata = sc.read_h5ad(os.path.join(base_path, "data/all_samples_raw_withTier2Annotation.h5ad"))

In [ ]:
np.unique(adata.obs['Annotation_Tier2'])

In [ ]:
# remove mixed marker clusters 
run_celltypes = ['Adipocytes', 'B cell', 'CD4 T cells', 'CD8 T cells',
       'CEACAM1 colonocyte-like', 'Cycing endothelium', 'Cycling Myeloid',
       'Cycling Stromal', 'Cycling T cells', 'Cycling plasma cell', 'DC',
       'Enteroendocrine-like', 'Fibroblast', 'Fibroblast-BMP5-SOX6',
       'Fibroblast-C3', 'Fibroblast-Infl', 'Fibroblast-KCNN3',
       'Fibroblast-MMP2-THY1', 'Germinal center / Cycling B cell',
       'Glial cells', 'HSP-hi - B cell', 'HSP-hi Myeloid',
       'HSP-hi Stromal', 'HSP-hi T cells', 'HSP-hi glial', 'ILCs',
       'LGR5 stem cell-like', 'Lymphatic endothelium',
       'MT-Ribo-hi Myeloid', 'MT-Ribo-hi Stromal', 'MT-Ribo-hi T cells',
       'MT-Ribo-hi endothelium', 'MT-Ribo-hi epithelial',
       'MUC2 goblet-like', 'Macrophage-Monocyte', 'Mast',
       'Myofibroblast-SMC', 'NK-Cytotoxic T cells', 'Neuronal cells',
       'Neutrophil', 'Patient-specific', 'Pericytes', 'Plasma cell',
       'Regulatory T cells', 'T helper cells', 'Vascular endothelium']
adata = adata[adata.obs['Annotation_Tier2'].isin(run_celltypes)]

In [ ]:
np.unique(adata.obs['Annotation_Tier2'])

In [ ]:
# list out the epithelial cell types 
np.unique(adata.obs['Annotation_Tier2'][adata.obs['Annotation_Tier1']=='Epithelial'])

In [ ]:
# Check sample that failed inferCNV 
np.unique(adata.obs['Annotation_Tier2'][adata.obs['FRID']=="COLFR0034_T1"])

In [ ]:
# Check sample that failed inferCNV - can see below it only has 2 non epi cells and both get removed 
adata.obs.loc[adata.obs['FRID'] == "COLFR0366_T1", 'Annotation_Tier1'].value_counts()

In [ ]:
# Create temporary annotation column for inferCNV that keeps epi subsets and makes everything else non-epi 
adata.obs['InferCNV_Annotation'] = 'Non-epithelial'
adata.obs.loc[adata.obs['Annotation_Tier2'] == 'CEACAM1 colonocyte-like', 'InferCNV_Annotation'] = 'CEACAM1 colonocyte-like'
adata.obs.loc[adata.obs['Annotation_Tier2'] == 'Enteroendocrine-like', 'InferCNV_Annotation'] = 'Enteroendocrine-like'
adata.obs.loc[adata.obs['Annotation_Tier2'] == 'LGR5 stem cell-like', 'InferCNV_Annotation'] = 'LGR5 stem cell-like'
adata.obs.loc[adata.obs['Annotation_Tier2'] == 'MT-Ribo-hi epithelial', 'InferCNV_Annotation'] = 'MT-Ribo-hi epithelial'
adata.obs.loc[adata.obs['Annotation_Tier2'] == 'MUC2 goblet-like', 'InferCNV_Annotation'] = 'MUC2 goblet-like'
adata.obs.loc[adata.obs['Annotation_Tier2'] == 'Patient-specific', 'InferCNV_Annotation'] = 'Patient-specific'
np.unique(adata.obs['InferCNV_Annotation'])

In [ ]:
# sanity check the re-assignment 
adata.obs.groupby(['InferCNV_Annotation', 'Annotation_Tier1']).size()

In [ ]:
# filter for cases when the number of cells in a cell type is too small 
sampleids = np.unique(adata.obs['FRID'])
cellCounts = adata.obs.groupby(['FRID', 'InferCNV_Annotation']).size()
cellCounts[cellCounts==1] #COLFR0034_T1 --> don't analyze this sample, there aren't enough cells 

In [ ]:
print(adata.obs.loc[adata.obs['FRID'] == 'COLFR0054_T1', 'InferCNV_Annotation'].value_counts())
print(adata.obs.loc[adata.obs['FRID'] == 'COLFR0123_T1', 'InferCNV_Annotation'].value_counts())
print(adata.obs.loc[adata.obs['FRID'] == 'COLFR0285_T1', 'InferCNV_Annotation'].value_counts())
print(adata.obs.loc[adata.obs['FRID'] == 'COLFR0301_T1', 'InferCNV_Annotation'].value_counts())
print(adata.obs.loc[adata.obs['FRID'] == 'COLFR0366_T1', 'InferCNV_Annotation'].value_counts())
print(adata.obs.loc[adata.obs['FRID'] == 'COLFR0374_T1', 'InferCNV_Annotation'].value_counts())
print(adata.obs.loc[adata.obs['FRID'] == 'COLFR0423_T1', 'InferCNV_Annotation'].value_counts())
print(adata.obs.loc[adata.obs['FRID'] == 'COLFR6101_T1', 'InferCNV_Annotation'].value_counts())
print(adata.obs.loc[adata.obs['FRID'] == 'COLFR6630_T1', 'InferCNV_Annotation'].value_counts())

In [ ]:
adata.shape

In [ ]:
# Remove the cell types that have a count of 1
adata = adata[~((adata.obs['FRID'] == 'COLFR0054_T1') & (adata.obs['InferCNV_Annotation'] == 'MUC2 goblet-like'))].copy()
adata = adata[~((adata.obs['FRID'] == 'COLFR0123_T1') & (adata.obs['InferCNV_Annotation'] == 'Enteroendocrine-like'))].copy()
adata = adata[~((adata.obs['FRID'] == 'COLFR0285_T1') & (adata.obs['InferCNV_Annotation'] == 'Enteroendocrine-like'))].copy()
adata = adata[~((adata.obs['FRID'] == 'COLFR0301_T1') & (adata.obs['InferCNV_Annotation'] == 'MUC2 goblet-like'))].copy()
adata = adata[~((adata.obs['FRID'] == 'COLFR0366_T1') & (adata.obs['InferCNV_Annotation'] == 'CEACAM1 colonocyte-like'))].copy()
adata = adata[~((adata.obs['FRID'] == 'COLFR0366_T1') & (adata.obs['InferCNV_Annotation'] == 'MT-Ribo-hi epithelial'))].copy()
adata = adata[~((adata.obs['FRID'] == 'COLFR0374_T1') & (adata.obs['InferCNV_Annotation'] == 'Enteroendocrine-like'))].copy()
adata = adata[~((adata.obs['FRID'] == 'COLFR0423_T1') & (adata.obs['InferCNV_Annotation'] == 'Enteroendocrine-like'))].copy()
adata = adata[~((adata.obs['FRID'] == 'COLFR6101_T1') & (adata.obs['InferCNV_Annotation'] == 'Enteroendocrine-like'))].copy()
adata = adata[~((adata.obs['FRID'] == 'COLFR6630_T1') & (adata.obs['InferCNV_Annotation'] == 'Enteroendocrine-like'))].copy()

In [ ]:
adata.shape

In [ ]:
len(np.unique(adata.obs['FRID']))

In [ ]:
sampleids = np.unique(adata.obs['FRID'])
for i in sampleids: 
    tmp = adata[adata.obs['FRID']==i]
    annot = tmp.obs['InferCNV_Annotation']
    counts =  pd.DataFrame(tmp.X.toarray())
    counts.index  = tmp.obs.index.tolist()
    counts.columns = tmp.var.index.tolist()
    counts = counts.T
    counts.to_csv(os.path.join(base_path, f'results/{date}_EOCRC_inferCNV/{i}_counts.tsv'), sep='\t', index=True)
    annot.to_csv(os.path.join(base_path, f'results/{date}_EOCRC_inferCNV/{i}_annotation.tsv'), sep='\t', index=True, header=False)

In [ ]:
# make .tsv file to upload to data table in Terra 
df = pd.DataFrame({
    'entity:sample_id': sampleids,
    'additional_args': ['--ref_group_names="Non-epithelial" --analysis_mode="subclusters" --HMM --denoise --cutoff=0.1 --leiden_function="modularity"'] * len(sampleids),
    'annotations_file': [f'gs://fc-ed878322-6cf6-49f5-8875-f9c237b931f2/results/2025-10-01_YOCRC_inferCNV/inferCNV_inputs/{i}_annotation.tsv' for i in sampleids],
    'gene_order_file': ['gs://fc-ed878322-6cf6-49f5-8875-f9c237b931f2/results/2025-10-01_YOCRC_inferCNV/inferCNV_inputs/Trinity_CTAT_cnv_hg38_gencode_v27.txt'] * len(sampleids),
    'raw_counts_matrix': [f'gs://fc-ed878322-6cf6-49f5-8875-f9c237b931f2/results/2025-10-01_YOCRC_inferCNV/inferCNV_inputs/{i}_counts.tsv' for i in sampleids],
})

In [ ]:
df.to_csv(os.path.join(base_path, 'results/{date}_EOCRC_inferCNV/data_table.tsv'), sep='\t', index=False, header=True)

In [ ]:
df